# Daume et al. (2024) on dandiset 000673: behaviour and single neurons

**Paper.** Daume J, Kaminski J, Schjetnan AGP, Salimpour Y, Khan U, Kyzar M, Reed CM, Anderson WS,
Valiante TA, Mamelak AN, Rutishauser U (2024). Control of working memory by phase-amplitude
coupling of human hippocampal neurons. *Nature* 629, 393-401.
[doi:10.1038/s41586-024-07309-z](https://doi.org/10.1038/s41586-024-07309-z).

**Dandiset.** [DANDI:000673](https://dandiarchive.org/dandiset/000673/0.240118.2135), version
0.240118.2135. The paper's reference 59 cites this version ("v.0.240118.2135"), so it is pinned,
as the protocol asks. Four later versions exist (0.240524.1751, 0.240524.1758, 0.250122.0110 and
the draft); in 0.250122.0110 every one of the 44 files has a new asset with a size a few hundred
kilobytes smaller, with the same paths. The analysis here does not use the later versions.

**Text the claims were written from.** The version of record (PMC11078732), with the Methods,
figure legends, Extended Data legends and Supplementary Information, and the paper's code
(github.com/rutishauserlab/SBCAT-release-NWB at commit a25e8cc3).

This notebook tests the claims on behaviour and single-neuron firing. The LFP claims are in
`reproduce_daume_2024_lfp.ipynb` and `reproduce_daume_2024_lfp_controls.ipynb`. Claims tested here:

| Claim | Tested here |
|---|---|
| cohort-counts | yes |
| fig1d-rt-load | yes |
| fig1d-accuracy-load | yes |
| behaviour-overall-accuracy | yes |
| category-cell-counts | yes |
| fig3b-persistent-activity | yes |
| edf4b-persistent-activity-by-area | yes |
| fig3c-category-load | yes |
| fig3d-category-correct-incorrect | yes |
| category-no-rt-effect | yes |
| edf8fgh-frontal-controls | panel h only; blocked (panels f and g need SFC, not run) |
| PAC neuron claims | in `reproduce_daume_2024_pac_neurons.ipynb` |
| fig3h-pac-category-firing | in `reproduce_daume_2024_lfp.ipynb` |
| SFC, noise correlation and decoding claims | blocked (see the attempt record) |

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import sbcat

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)
warnings.filterwarnings("ignore", category=RuntimeWarning)

started = time.time()
assets = sbcat.list_assets()
S = sbcat.load_all(assets)
print(f"loaded {len(S)} files of dandiset {sbcat.DANDISET_ID} version {sbcat.DANDISET_VERSION} in {time.time() - started:.0f} s")

## Data check

In [ ]:
info = pd.DataFrame([{
    "path": s["path"], "subject": s["subject"], "session": s["session"], "age": s["age"], "sex": s["sex"],
    "trials": len(s["trials"]), "load1": int((s["trials"].load == 1).sum()), "load3": int((s["trials"].load == 3).sum()),
    "units": len(s["units"]), "lfp_channels": len(s["lfp_electrodes"]), "electrodes": len(s["electrodes"]),
    "lfp_rate": s["lfp_info"]["rate"] if s["lfp_info"] else np.nan,
    "lfp_unit": s["lfp_info"]["unit"] if s["lfp_info"] else None} for s in S])
print(info.to_string(index=False))
print("\nsessions:", len(info), "; patients:", info.subject.nunique(),
      "; sessions per patient:", info.subject.value_counts().value_counts().sort_index().to_dict())
pat = info.drop_duplicates("subject")
ages = pat.age.str.strip("PY").astype(int)
print("patients by sex:", pat.sex.value_counts().to_dict(), f"; age {ages.mean():.2f} +/- {ages.std(ddof=1):.2f} years")
print("trials per session:", info.trials.value_counts().to_dict())
print("files without LFP:", info.path[info.lfp_channels == 0].tolist())
print("LFP description:", {s["lfp_info"]["description"] for s in S if s["lfp_info"]})

In [ ]:
units = pd.concat([s["units"].assign(path=s["path"], subject=s["subject"]) for s in S], ignore_index=True)
lfp = pd.concat([s["electrodes"].iloc[s["lfp_electrodes"]].assign(path=s["path"]) for s in S], ignore_index=True)
print("electrode locations:", sorted(set(units.location) | set(lfp.location)))
print("\nunits by area and hemisphere:\n", units.pivot_table(index="area", columns="hemisphere", values="unit", aggfunc="count").to_string())
print("\nLFP channels by area and hemisphere:\n", lfp.pivot_table(index="area", columns="hemisphere", values="x", aggfunc="count").to_string())
# Each unit's electrode, and each LFP channel, is a row of the electrodes table; electrodes of one
# area and hemisphere in a session share coordinates (one bundle of 8 wires).
bundles = pd.concat([s["electrodes"].assign(path=s["path"]) for s in S]).groupby(["path", "location"]).agg(
    wires=("x", "size"), distinct_xyz=("x", lambda v: len(set(v))))
print("\nwires per bundle (area and hemisphere of a session):", bundles.wires.value_counts().sort_index().to_dict(),
      "; bundles with more than one coordinate:", int((bundles.distinct_xyz > 1).sum()), "of", len(bundles))
# Columns of the units table (for the waveform measure of Extended Data Fig. 8d,e)
ucols = pd.Series({s["path"]: tuple(sorted(s["units_columns"])) for s in S})
print("\nunits table columns, by the number of files that have them:", ucols.value_counts().to_dict())
wf = [s["units_columns"].get("waveform_mean") for s in S]
print("waveform_mean shapes (first 3 files):", wf[:3], "; files without it:", sum(w is None for w in wf))

In [ ]:
tr_all = pd.concat([s["trials"].assign(path=s["path"]) for s in S], ignore_index=True)
d = tr_all
print("trial columns:", [c for c in S[0]["trials"].columns])
print("loads:", d.load.value_counts().to_dict(), "; response_accuracy:", d.response_accuracy.value_counts().to_dict(),
      "; probe_in_out:", d.probe_in_out.value_counts().to_dict())
dur = pd.DataFrame({
    "fixation_s": d.timestamps_Encoding1 - d.timestamps_FixationCross,
    "picture1_s": d.timestamps_Encoding1_end - d.timestamps_Encoding1,
    "maintenance_s": d.timestamps_Probe - d.timestamps_Maintenance,
    "rt_s": d.rt})
print("\n", dur.describe().round(3).to_string())
l3 = d[d.load == 3]
gap = pd.concat([l3.timestamps_Encoding2 - l3.timestamps_Encoding1_end, l3.timestamps_Encoding3 - l3.timestamps_Encoding2_end])
print("\nload 3 blank between pictures (s): min %.3f, max %.3f (Methods: 0.017 to 0.200)" % (gap.min(), gap.max()))
print("maintenance onset equals the end of the last picture:",
      bool(np.allclose(np.where(d.load == 1, d.timestamps_Encoding1_end, d.timestamps_Encoding3_end), d.timestamps_Maintenance)))
cats = pd.concat([d[f"cat_{k}"] for k in ["Encoding1", "Encoding2", "Encoding3", "Probe"]])
print("category codes (first digit of the picture ID):", cats.value_counts().sort_index().to_dict())
l3c = l3[["cat_Encoding1", "cat_Encoding2", "cat_Encoding3"]].to_numpy()
print("load 3 trials with three different categories:", int((np.array([len(set(r)) for r in l3c]) == 3).sum()), "of", len(l3c))
match = d.PicIDs_Probe.to_numpy()[:, None] == d[["PicIDs_Encoding1", "PicIDs_Encoding2", "PicIDs_Encoding3"]].to_numpy()
print("probe_in_out equals 'probe is one of the encoded pictures':", bool(np.all(match.any(1) == (d.probe_in_out == 1))))
print("RT above 10 s:", int((d.rt > 10).sum()), "trials; max %.1f s" % d.rt.max())

In [ ]:
# The TTL events against the trials table: every trial timestamp should be an event with the
# code the file's description gives (11 fixation, 1-3 pictures, 6 maintenance, 7 probe, 8 response).
codes = {"timestamps_FixationCross": 11, "timestamps_Encoding1": 1, "timestamps_Encoding2": 2, "timestamps_Encoding3": 3,
         "timestamps_Maintenance": 6, "timestamps_Probe": 7, "timestamps_Response": 8}
rows = []
for s in S:
    ev = s["events"]
    for col, code in codes.items():
        t = s["trials"][col].to_numpy()
        t = t[t != 0]
        evt = np.sort(ev.t[ev.code == code].to_numpy())
        i = np.clip(np.searchsorted(evt, t), 1, len(evt) - 1) if len(evt) > 1 else np.zeros(len(t), int)
        dist = np.minimum(np.abs(evt[i] - t), np.abs(evt[i - 1] - t)) if len(evt) > 1 else np.full(len(t), np.inf)
        rows.append({"path": s["path"], "column": col, "n": len(t), "max_offset_ms": 1000 * dist.max()})
evchk = pd.DataFrame(rows)
print(evchk.groupby("column").max_offset_ms.describe().round(3).to_string())
print("sessions with a trial timestamp more than 1 ms from its TTL event:",
      sorted(set(evchk.path[evchk.max_offset_ms > 1])))

In [ ]:
# Spikes against the trial clock: spike times span the task, and spikes respond to picture onset.
rows = []
for s in S:
    allsp = np.concatenate([sp for sp in s["spikes"] if len(sp)]) if len(s["spikes"]) else np.array([])
    tr = s["trials"]
    rows.append({"path": s["path"], "spike_min": allsp.min(), "spike_max": allsp.max(),
                 "task_start": tr.start_time.min(), "task_end": tr.stop_time.max(),
                 "units_with_unsorted_spike_times": len(s["unsorted_units"])})
spk = pd.DataFrame(rows)
print(spk.round(1).to_string(index=False))
print("units whose stored spike times are not sorted (sorted on loading), with the largest backward step (s):",
      {s["path"]: s["unsorted_units"] for s in S if s["unsorted_units"]})
rates_all = np.array([len(sp) / (s["trials"].stop_time.max() - s["trials"].start_time.min()) for s in S for sp in s["spikes"]])
print("mean rate over the task per unit (Hz): median %.2f, range %.3f to %.1f" % (np.median(rates_all), rates_all.min(), rates_all.max()))

In [ ]:
# Clock check by contrast: across MTL units, the share with a category effect (one-way ANOVA on
# the 0.2-1.0 s window after picture onset, p < 0.05) at the true onsets and with onsets shifted.
shifts = [-3.0, -1.5, -0.6, 0.0, 0.6, 1.5, 3.0]
res = {sh: [] for sh in shifts}
for s in S:
    ts, cats_ = sbcat.presentations(s)
    for u in np.where(s["units"].area.isin(sbcat.MTL))[0]:
        for sh in shifts:
            r = sbcat.count_in(s["spikes"][u], ts + sh + 0.2, ts + sh + 1.0)
            groups = [r[cats_ == c] for c in np.unique(cats_)]
            res[sh].append(stats.f_oneway(*groups).pvalue if r.std() > 0 else 1.0)
print("share of MTL units with a category effect, by shift of the picture onsets (s):")
print({sh: round(float(np.mean(np.array(v) < 0.05)), 3) for sh, v in res.items()})

In [ ]:
# Recordings deposited twice: compare spike trains and trial tables across files.
sig = []
for s in S:
    first = tuple(np.round(np.concatenate([sp[:5] for sp in s["spikes"]]), 4)[:50])
    sig.append((s["path"], tuple(np.round(s["trials"].rt.to_numpy()[:20], 4)), first))
dups = [(a[0], b[0]) for i, a in enumerate(sig) for b in sig[i + 1:] if a[1] == b[1] or a[2] == b[2]]
print("pairs of files with identical RT sequences or identical first spike times:", dups if dups else "none")

All counts match the paper (checked under cohort-counts below). The trials table agrees with the
TTL events, the maintenance period starts at the end of the last picture and lasts 2.53 to 3.20 s
(the Methods say 2.55 to 2.85 s), load 3 trials hold three categories, and `probe_in_out` is
whether the probe was one of the encoded pictures. The share of MTL units with a category effect
peaks at the true picture onsets and falls with the shift, so the spike times share the trial
clock (consistent with the data; the shift contrast shows a response locked to the onsets).

## cohort-counts

> The study recorded 36 patients in 44 sessions, with 1,454 single neurons (360 hippocampus, 496
> amygdala, 204 pre-SMA, 188 dACC, 206 vmPFC) and 1,922 microwire LFP channels (586, 421, 283, 325
> and 307).

In [ ]:
paper_units = {"hippocampus": 360, "amygdala": 496, "preSMA": 204, "dACC": 188, "vmPFC": 206}
paper_lfp = {"hippocampus": 586, "amygdala": 421, "preSMA": 283, "dACC": 325, "vmPFC": 307}
cc = pd.DataFrame({"units": units.area.value_counts(), "paper_units": pd.Series(paper_units),
                   "lfp": lfp.area.value_counts(), "paper_lfp": pd.Series(paper_lfp)}).loc[sbcat.AREAS]
print(cc.to_string())
print("totals: sessions %d (44), patients %d (36), units %d (1454), LFP channels %d (1922)" % (
    len(info), info.subject.nunique(), len(units), len(lfp)))
print("units per patient: min %d, median %d, max %d" % tuple(units.groupby("subject").size().agg(["min", "median", "max"])))

**Result: reproduced.** 44 sessions, 36 patients, 1,454 units and 1,922 LFP channels, and every
per-area count equals the paper's.

## fig1d-rt-load and fig1d-accuracy-load

> Patients responded more slowly in load 3 than in load 1 trials (1.46 s against 1.33 s).

> Patients were less accurate in load 3 than in load 1 trials (91.60% against 95.71%).

In [ ]:
rng = np.random.default_rng(101)
beh = []
for s in S:
    tr = s["trials"]
    row = {"path": s["path"], "subject": s["subject"]}
    for L in (1, 3):
        t = tr[tr.load == L]
        row[f"acc{L}"] = 100 * t.correct.mean()
        row[f"rt_median{L}"] = t.rt[t.correct].median()
        row[f"rt_mean{L}"] = t.rt[t.correct].mean()
    beh.append(row)
beh = pd.DataFrame(beh)
for kind in ["rt_median", "rt_mean"]:
    r = sbcat.perm_ttest_paired(beh[f"{kind}3"], beh[f"{kind}1"], rng)
    print(f"{kind}: load 1 {beh[f'{kind}1'].mean():.3f} s, load 3 {beh[f'{kind}3'].mean():.3f} s; "
          f"t{r['df']} = {r['t']:.2f}, permutation p = {r['p']:.4f}; sessions with load 3 slower: "
          f"{int((beh[f'{kind}3'] > beh[f'{kind}1']).sum())} of {len(beh)}")
print("range of session median RT (s): %.2f to %.2f" % (beh[["rt_median1", "rt_median3"]].min().min(), beh[["rt_median1", "rt_median3"]].max().max()))
by_pat = beh.groupby("subject")[["rt_median1", "rt_median3"]].mean()
print("patients with load 3 slower (median RT, sessions averaged):", int((by_pat.rt_median3 > by_pat.rt_median1).sum()), "of", len(by_pat))

In [ ]:
rng = np.random.default_rng(102)
r = sbcat.perm_ttest_paired(beh.acc3, beh.acc1, rng)
print(f"accuracy: load 1 {beh.acc1.mean():.2f}%, load 3 {beh.acc3.mean():.2f}%; t{r['df']} = {r['t']:.2f}, permutation p = {r['p']:.4f}")
print("sessions with load 3 less accurate:", int((beh.acc3 < beh.acc1).sum()), "; equal:", int((beh.acc3 == beh.acc1).sum()), "of", len(beh))
by_pat = beh.groupby("subject")[["acc1", "acc3"]].mean()
print("patients with load 3 less accurate:", int((by_pat.acc3 < by_pat.acc1).sum()), "of", len(by_pat))
fig, ax = plt.subplots(1, 2, figsize=(7, 3))
for a, k, lab in [(ax[0], "rt_median", "median RT (s)"), (ax[1], "acc", "accuracy (%)")]:
    a.plot([1, 3], beh[[f"{k}1", f"{k}3"]].T, color="0.7", lw=0.8)
    a.plot([1, 3], beh[[f"{k}1", f"{k}3"]].mean(), "k-o")
    a.set_xticks([1, 3]); a.set_xlabel("load"); a.set_ylabel(lab)
plt.tight_layout(); plt.show()

**Result: reproduced (RT).** The mean over sessions of the median RT of correct trials is 1.233 s in
load 1 and 1.360 s in load 3, both within tolerance (1.33 and 1.46 s, 10%), and the paired
permutation test gives t43 = 5.33, p = 0.0001 with load 3 slower. With the mean RT per session
(variant) load 1 is 1.633 s and load 3 1.596 s (p = 0.66); RTs reach 74.2 s, with 69 trials
above 10 s.

**Result: reproduced (accuracy).** 95.10% in load 1 and 91.02% in load 3, within tolerance of
95.71% and 91.60%, with t43 = -5.24, p = 0.0001 and load 3 less accurate.

## behaviour-overall-accuracy

> Mean accuracy was 93.66 +/- 7.04% (s.d. over sessions), and 78.34 +/- 20.09% of errors were
> false negatives.

In [ ]:
acc = np.array([100 * s["trials"].correct.mean() for s in S])
fn = []
for s in S:
    tr = s["trials"]
    err = tr[~tr.correct]
    if len(err):
        fn.append(100 * (err.probe_in_out == 1).mean())
fn = np.array(fn)
print(f"accuracy over sessions: {acc.mean():.2f} +/- {acc.std(ddof=1):.2f}% (range {acc.min():.1f} to {acc.max():.1f})")
print(f"false negatives as a share of errors: {fn.mean():.2f} +/- {fn.std(ddof=1):.2f}% over {len(fn)} sessions with errors")
allerr = tr_all[~tr_all.correct]
print(f"pooled over all errors: {100 * (allerr.probe_in_out == 1).mean():.2f}% of {len(allerr)} errors")

**Result: reproduced.** Mean accuracy is 93.05 +/- 7.22% (paper 93.66 +/- 7.04%), and false
negatives are 76.08 +/- 20.11% of errors over the 41 sessions with errors (paper 78.34 +/-
20.09%); both are within tolerance.

## category-cell-counts

> Category neurons were found above chance in the hippocampus (89, 24.72%), amygdala (181,
> 36.49%) and vmPFC (37, 17.96%), but not in the pre-SMA and dACC.

In [ ]:
t0 = time.time()
cat_table, cat_null = sbcat.category_table(S, seed=103)
print(f"selection and 500 label shuffles in {time.time() - t0:.0f} s")
expected = {"hippocampus": 89, "amygdala": 181, "preSMA": 18, "dACC": 14, "vmPFC": 37}
rows = []
for a in sbcat.AREAS:
    m = (cat_table.area == a).to_numpy()
    obs = int(cat_table.is_category[m].sum())
    null = cat_null[m].sum(0)
    rows.append({"area": a, "neurons": int(m.sum()), "category": obs, "percent": 100 * obs / m.sum(),
                 "paper": expected[a], "within_10pct": abs(obs - expected[a]) <= 0.1 * expected[a],
                 "null_mean": null.mean(), "null_p99": np.percentile(null, 99),
                 "p_shuffle": (np.sum(null >= obs) + 1) / (len(null) + 1), "above_p99": obs > np.percentile(null, 99)})
cat_counts = pd.DataFrame(rows)
print(cat_counts.round(3).to_string(index=False))
print("\ncategory neurons per patient (MTL):")
print(cat_table[cat_table.area.isin(sbcat.MTL)].groupby("subject").is_category.agg(["sum", "size"]).T.to_string())
print("\npreferred categories of MTL category neurons:",
      cat_table[cat_table.is_category & cat_table.area.isin(sbcat.MTL)].preferred.value_counts().sort_index().to_dict())
# How the count depends on the post hoc threshold (one-sided 0.05 as two-sided 0.1, the code's)
for thr in [0.05, 0.1]:
    c = cat_table[(cat_table.p_anova < 0.05) & (cat_table.p_posthoc < thr)].area.value_counts().reindex(sbcat.AREAS)
    print(f"post hoc two-sided p < {thr}:", c.to_dict())

**Result: partially reproduced.** Every count is within 10% of the paper's (hippocampus 94, amygdala
181, vmPFC 40, pre-SMA 18, dACC 13), and the hippocampus, amygdala and vmPFC exceed the 99th
percentile of the shuffle null (p = 0.002 each). The dACC does not (p = 0.088), but the pre-SMA's
18 also exceeds its 99th percentile of 17 (p = 0.010), where the paper reports no excess. The
pre-SMA sits at the boundary: 18 is one above the 99th percentile, and p = 0.010 equals the 0.01
threshold, so with 500 shuffles the two forms of the rule do not separate it from chance clearly.

## Maintenance rates of category neurons

Per neuron, the mean rate during maintenance (0 to 2.5 s) over correct preferred trials
(preferred category among the encoded pictures, G4) and correct non-preferred trials, and the
baseline (-0.9 to -0.3 s before picture 1) over all correct trials. Reading (a) of O6 uses Hz;
reading (b) the percentage change from each neuron's baseline (neurons with a baseline of 0 Hz
are left out under (b)).

In [ ]:
cat_units = cat_table[cat_table.is_category].reset_index(drop=True)
cat_rates = sbcat.trial_rates(S, cat_units)


def neuron_means(rates, cond_fn, reading):
    """Mean maintenance value per neuron over the trials cond_fn selects, in Hz or % of baseline."""
    out = []
    for r in rates:
        c = cond_fn(r)
        base = r["base"][r["correct"]].mean()
        v = r["maint"][c].mean() if c.sum() else np.nan
        if reading == "b":
            v = 100 * (v - base) / base if base > 0 else np.nan
        out.append(v)
    return np.array(out)


def base_means(rates, reading):
    if reading == "b":
        return np.array([0.0 if r["base"][r["correct"]].mean() > 0 else np.nan for r in rates])
    return np.array([r["base"][r["correct"]].mean() for r in rates])


def persistence_tests(sel, rng):
    rows = []
    for reading in ["a", "b"]:
        rs = [cat_rates[i] for i in np.where(sel)[0]]
        pref = neuron_means(rs, lambda r: r["correct"] & r["pref"], reading)
        npref = neuron_means(rs, lambda r: r["correct"] & ~r["pref"], reading)
        base = base_means(rs, reading)
        for name, x, y in [("preferred vs baseline", pref, base), ("non-preferred vs baseline", npref, base),
                           ("preferred vs non-preferred", pref, npref)]:
            t = sbcat.perm_ttest_paired(x, y, rng)
            rows.append({"reading": reading, "test": name, "n": t["n"], "mean_x": t["mean_x"], "mean_y": t["mean_y"],
                         "t": t["t"], "p": t["p"]})
    return pd.DataFrame(rows)

## fig3b-persistent-activity

> MTL category neurons stayed above baseline during maintenance in both preferred and
> non-preferred trials, and fired more when their preferred category was held in memory.

In [ ]:
rng = np.random.default_rng(104)
mtl = cat_units.area.isin(sbcat.MTL).to_numpy()
fig3b = persistence_tests(mtl, rng)
print(f"MTL category neurons: {mtl.sum()} (paper 270)")
print(fig3b.round(4).to_string(index=False))
print("neurons left out under (b), baseline 0 Hz:", int(np.isnan(base_means([cat_rates[i] for i in np.where(mtl)[0]], 'b')).sum()))
d_pp = neuron_means([cat_rates[i] for i in np.where(mtl)[0]], lambda r: r["correct"] & r["pref"], "a") - \
    neuron_means([cat_rates[i] for i in np.where(mtl)[0]], lambda r: r["correct"] & ~r["pref"], "a")
per_pat = pd.Series(d_pp).groupby(cat_units.subject[mtl].to_numpy()).mean()
print("patients with preferred above non-preferred (mean over their neurons, Hz):", int((per_pat > 0).sum()), "of", len(per_pat))
print("range of the per-neuron preferred minus non-preferred difference (Hz): %.2f to %.2f" % (np.nanmin(d_pp), np.nanmax(d_pp)))

**Result: partially reproduced.** Over 275 MTL category neurons (paper 270), maintenance rates
exceed baseline in preferred and non-preferred trials under both readings (p = 0.0001 each). The
preferred against non-preferred test gives p = 0.0013 in % of baseline (b) but p = 0.072 in Hz
(a), so the criterion (all three tests under both readings) holds only in part. Under reading (b)
alone, the reading adopted for the claims that leave the choice open, all three tests hold.

## edf4b-persistent-activity-by-area

> Persistent and selective maintenance activity of category neurons held in the hippocampus and
> amygdala separately, but vmPFC category neurons returned to baseline and did not differ between
> preferred and non-preferred trials.

In [ ]:
rng = np.random.default_rng(105)
tabs = []
for a in ["hippocampus", "amygdala", "vmPFC"]:
    sel = (cat_units.area == a).to_numpy()
    t = persistence_tests(sel, rng)
    t.insert(0, "area", a)
    tabs.append(t)
edf4b = pd.concat(tabs, ignore_index=True)
print(edf4b.round(4).to_string(index=False))

**Result: partially reproduced.** In % of baseline (b) the hippocampus (94 neurons) and amygdala
(181) pass all three tests (p <= 0.0385) and the vmPFC (40) passes none (p >= 0.064). In Hz (a)
the vmPFC again passes none, but the preferred against non-preferred test fails in the
hippocampus (p = 0.24) and amygdala (p = 0.18).

## fig3c-category-load

> In preferred trials, MTL category neurons fired more during maintenance in load 1 than in load
> 3, while no load effect appeared in non-preferred trials.

In [ ]:
rng = np.random.default_rng(106)
rs = [cat_rates[i] for i in np.where(mtl)[0]]
rows = []
for reading in ["a", "b"]:
    for name, pref in [("preferred", True), ("non-preferred", False)]:
        x1 = neuron_means(rs, lambda r: r["correct"] & (r["pref"] == pref) & (r["load"] == 1), reading)
        x3 = neuron_means(rs, lambda r: r["correct"] & (r["pref"] == pref) & (r["load"] == 3), reading)
        t = sbcat.perm_ttest_paired(x1, x3, rng)
        rows.append({"reading": reading, "trials": name, "n": t["n"], "load1": t["mean_x"], "load3": t["mean_y"], "t": t["t"], "p": t["p"]})
fig3c = pd.DataFrame(rows)
print(fig3c.round(4).to_string(index=False))

**Result: not reproduced, resting on an assumption.** In % of baseline (b), the reading adopted (the
legends show rates as % change from baseline and give the P values with those panels), preferred
trials do not differ between loads (35.29 against 30.85%, t = 0.47, p = 0.77). In Hz (a) load 1
is higher in preferred trials (2.90 against 2.78 Hz, t = 2.20, p = 0.022) and non-preferred trials
do not differ (p = 0.20), which would reproduce the claim.

## fig3d-category-correct-incorrect

> MTL category neurons fired more during maintenance in correct than in incorrect trials.

Readings: trials (a) all, (b) preferred only; minimum incorrect trials per neuron 1, 2 or 3
(O7); rates in Hz or % of baseline (O6).

In [ ]:
rng = np.random.default_rng(107)
rows = []
for trials in ["all", "preferred"]:
    for min_inc in [1, 2, 3]:
        for reading in ["a", "b"]:
            keep = []
            for r in rs:
                base = (r["pref"] if trials == "preferred" else np.ones(len(r["pref"]), bool))
                keep.append(((~r["correct"]) & base).sum() >= min_inc)
            keep = np.array(keep)
            rk = [r for r, k in zip(rs, keep) if k]
            tsel = (lambda r: r["pref"]) if trials == "preferred" else (lambda r: np.ones(len(r["pref"]), bool))
            xc = neuron_means(rk, lambda r: r["correct"] & tsel(r), reading)
            xi = neuron_means(rk, lambda r: ~r["correct"] & tsel(r), reading)
            t = sbcat.perm_ttest_paired(xc, xi, rng)
            rows.append({"trials": trials, "min_incorrect": min_inc, "reading": reading, "n": t["n"],
                         "correct": t["mean_x"], "incorrect": t["mean_y"], "t": t["t"], "p": t["p"]})
fig3d = pd.DataFrame(rows)
print(fig3d.round(4).to_string(index=False))
print("paper: n = 246, t245 = 2.43, p = 0.02")

**Result: reproduced, resting on an assumption.** In % of baseline (the adopted reading), over all
trials, the 251 neurons with at least one incorrect trial (paper 246) fire more in correct than
incorrect trials (26.56 against 9.27%, t = 2.50, p = 0.0035); with at least two incorrect trials p
= 0.010, with three p = 0.055. In Hz the test fails (p >= 0.12), as it does on preferred trials
only (p >= 0.10).

## category-no-rt-effect

> Category neuron maintenance rates did not differ between fast and slow RT trials in the
> hippocampus or amygdala.

In [ ]:
rng = np.random.default_rng(108)
rows = []
for a in sbcat.MTL:
    sel = np.where((cat_units.area == a).to_numpy())[0]
    for reading in ["a", "b"]:
        fast_v, slow_v = [], []
        for i in sel:
            r = cat_rates[i]
            base = r["base"][r["correct"]].mean()
            use = r["correct"] & r["pref"]
            fast, slow = sbcat.rt_split(r["load"], r["rt"], use)
            fv = np.nanmean([r["maint"][fast & (r["load"] == L)].mean() for L in (1, 3)])
            sv = np.nanmean([r["maint"][slow & (r["load"] == L)].mean() for L in (1, 3)])
            if reading == "b":
                fv, sv = (100 * (fv - base) / base, 100 * (sv - base) / base) if base > 0 else (np.nan, np.nan)
            fast_v.append(fv); slow_v.append(sv)
        t = sbcat.perm_ttest_paired(fast_v, slow_v, rng)
        rows.append({"area": a, "reading": reading, "n": t["n"], "fast": t["mean_x"], "slow": t["mean_y"], "t": t["t"], "p": t["p"]})
rt_tab = pd.DataFrame(rows)
print(rt_tab.round(4).to_string(index=False))
print("paper: hippocampus t88 = 0.73, p = 0.47; amygdala t180 = 1.38, p = 0.18")

**Result: reproduced.** No area and reading gives p < 0.05 (hippocampus p = 0.85 and 0.22,
amygdala p = 0.098 and 0.51).

## edf8fgh-frontal-controls (panel h only)

> Across all vmPFC neurons, neither within-region SFC nor SFC to hippocampal LFPs differed between
> loads, and maintenance firing rates of all neurons in the pre-SMA, dACC and vmPFC did not
> differ between correct and incorrect trials.

Panels f and g need spike-field coherence, which this attempt did not run, so the claim is
blocked; panel h is shown here for the record.

In [ ]:
rng = np.random.default_rng(109)
front = cat_table[cat_table.area.isin(["preSMA", "dACC", "vmPFC"])].reset_index(drop=True)
front_rates = sbcat.trial_rates(S, front)
rows = []
for a, n_paper in [("preSMA", 201), ("dACC", 180), ("vmPFC", 201)]:
    sel = np.where((front.area == a).to_numpy())[0]
    for min_inc in [1, 2, 3]:
        for reading in ["a", "b"]:
            rk = [front_rates[i] for i in sel if (~front_rates[i]["correct"]).sum() >= min_inc]
            xc = neuron_means(rk, lambda r: r["correct"], reading)
            xi = neuron_means(rk, lambda r: ~r["correct"], reading)
            t = sbcat.perm_ttest_paired(xc, xi, rng)
            rows.append({"area": a, "paper_n": n_paper, "min_incorrect": min_inc, "reading": reading, "n": t["n"],
                         "correct": t["mean_x"], "incorrect": t["mean_y"], "t": t["t"], "p": t["p"]})
print(pd.DataFrame(rows).round(4).to_string(index=False))

**Result: blocked** (panels f and g need spike-field coherence, not run). Panel h: in % of baseline
(the adopted reading of O6) with at least one incorrect trial, the n are the paper's exactly (pre-SMA
201, dACC 180, vmPFC 201) and no area differs (p = 0.54, 0.71, 0.97); in Hz with at least one
incorrect trial the pre-SMA differs (incorrect 3.07 against correct 2.91 Hz, p = 0.039), with two
or three incorrect trials it does not (p = 0.089, 0.11). That the % of baseline reading gives the
paper's n in all three areas (neurons with no baseline spike drop out) supports it as the reading
of the paper's firing rate tests.

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| cohort-counts | reproduced |
| fig1d-rt-load | reproduced |
| fig1d-accuracy-load | reproduced |
| behaviour-overall-accuracy | reproduced |
| category-cell-counts | partially reproduced (pre-SMA also above chance) |
| fig3b-persistent-activity | partially reproduced (selectivity only in % of baseline) |
| edf4b-persistent-activity-by-area | partially reproduced |
| fig3c-category-load | not reproduced, resting on an assumption (% of baseline) |
| fig3d-category-correct-incorrect | reproduced, resting on an assumption (% of baseline) |
| category-no-rt-effect | reproduced |
| edf8fgh-frontal-controls | blocked (SFC not run; panel h holds in % of baseline) |

The LFP claims are in `reproduce_daume_2024_lfp.ipynb` and `reproduce_daume_2024_lfp_controls.ipynb`, and the PAC neuron claims in
`reproduce_daume_2024_pac_neurons.ipynb`; the SFC, noise correlation and decoding claims are
blocked (attempt record).

How the analysis differs from the paper's: the category selection follows the paper's code, with
the shuffled-label null drawn by shuffling the labels of a session's presentations together; the
permutation tests flip the signs of paired differences (10,000 draws); whether rates are compared
in Hz or in % of baseline is not stated, both are shown, and % of baseline is adopted where they
disagree; units recorded in several sessions of
one patient are counted once per session, as the paper's totals do.

What was found about the dataset: every count of the paper is in it; the trials table agrees with
the TTL events; spike times are unsorted in four units of two files (three in sub-13 jump back by
about 1,320 s, as if two spike trains were concatenated); the maintenance period lasts 2.53 to
3.20 s, against 2.55 to 2.85 s in the Methods; and RTs reach 74 s. The units table holds waveforms (`waveforms`, and `waveform_mean` of 256
samples in 43 of the 44 files).

In [ ]:
print(f"notebook ran in {(time.time() - started) / 60:.1f} min")